### 파일 읽기 방법 서버랑 통일

In [ ]:
import cv2
import numpy as np
from utils.Mediapipe import MediaPipe
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_PATH = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/_origin_data/028_data/data/1. 디지털카메라/0001/0001_01_F.jpg'

def crop_cheeks_from_detection_result(landmarks, mp_image):
    image = mp_image.numpy_view()
    # image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape
    def to_px(lm): return int(lm.x * w), int(lm.y * h)

    point_0 = to_px(landmarks[0])
    point_264 = to_px(landmarks[264])
    point_34 = to_px(landmarks[34])

    left_cheek_x1, left_cheek_y1 = min(point_0[0], point_264[0]), min(
        point_0[1], point_264[1]
    )
    left_cheek_x2, left_cheek_y2 = max(point_0[0], point_264[0]), max(
        point_0[1], point_264[1]
    )
    left_cheek = image[left_cheek_y1:left_cheek_y2, left_cheek_x1:left_cheek_x2]

    right_cheek_x1, right_cheek_y1 = min(point_0[0], point_34[0]), min(
        point_0[1], point_34[1]
    )
    right_cheek_x2, right_cheek_y2 = max(point_0[0], point_34[0]), max(
        point_0[1], point_34[1]
    )
    right_cheek = image[
        right_cheek_y1:right_cheek_y2, right_cheek_x1:right_cheek_x2
    ]

    return left_cheek, right_cheek

face_landmarker = MediaPipe()

print(f"이미지 로딩: {IMAGE_PATH}")
with open(IMAGE_PATH, 'rb') as f:
    image_content = f.read()

mp_image = face_landmarker.create_mp_image(image_content)

detection_result = face_landmarker.landmarker.detect(mp_image)
landmarks = detection_result.face_landmarks[0]

left_cheek, right_cheek = crop_cheeks_from_detection_result(landmarks, mp_image)

# 저장은 png로 해야 안깨진다
left_path = '/home/work/hocheol_dir/workspace/tmp/left.png'
right_path = '/home/work/hocheol_dir/workspace/tmp/right.png'
cv2.imwrite(left_path, cv2.cvtColor(left_cheek, cv2.COLOR_RGB2BGR))
cv2.imwrite(right_path, cv2.cvtColor(right_cheek, cv2.COLOR_RGB2BGR))

del face_landmarker
del mp_image
del detection_result
del landmarks
del left_cheek
del right_cheek

In [ ]:
face_landmarker = MediaPipe()
with open(IMAGE_PATH, 'rb') as f:
    image_content = f.read()
mp_image = face_landmarker.create_mp_image(image_content)
detection_result = face_landmarker.landmarker.detect(mp_image)
landmarks = detection_result.face_landmarks[0]
left_cheek, right_cheek = crop_cheeks_from_detection_result(landmarks, mp_image)

with open(left_path, 'rb') as f:
    left_cheek2 = face_landmarker.create_mp_image(f.read()).numpy_view()
with open(right_path, 'rb') as f:
    right_cheek2 = face_landmarker.create_mp_image(f.read()).numpy_view()

print(f"crop 후 png로 저장하면 많이 안깨지니? : {np.array_equal(left_cheek, left_cheek2)}")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 10))
ax[0, 0].imshow(left_cheek2)
ax[0, 0].set_title('Left Cheek2')
ax[0, 1].imshow(right_cheek2)
ax[0, 1].set_title('Right Cheek2')
ax[1, 0].imshow(left_cheek)
ax[1, 0].set_title('Left Cheek')
ax[1, 1].imshow(right_cheek)
ax[1, 1].set_title('Right Cheek')
plt.tight_layout()
plt.show()

### 028

01, 02 데이터만 사용<br>
01 : F, L15, L30, R15, R30 존재<br>
02 : F, L, R 존재

028에서 F 먼저 하고 L, R 하자

In [ ]:
import os
from utils.Mediapipe import MediaPipe

source_dirs = ['/home/work/hocheol_dir/workspace/datasets/_origin_data/028_data/data/1. 디지털카메라', '/home/work/hocheol_dir/workspace/datasets/_origin_data/028_data/data/2. 스마트패드']
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/028_data/data'

os.makedirs(dest_dir, exist_ok=True)

face_landmarker = MediaPipe()

path_groups = []

for source_dir in source_dirs:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith('_F.jpg'):
                source_path = os.path.join(root, file)
                left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
                right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
                path_groups.append((source_path, left_path, right_path))

path_groups.sort()
print(len(path_groups))

In [ ]:
# 정면
import sys
sys.path.insert(0, '/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_one
from tqdm import tqdm
import multiprocessing
from multiprocessing import Pool


multiprocessing.set_start_method('spawn', force=True)
num_workers = 64

with Pool(num_workers) as pool:
    fail_list = list(tqdm(pool.imap(process_one, path_groups), total=len(path_groups)))

fail_paths = set(filter(None, fail_list))
print(f"Failed to process {len(fail_paths)} images.")

In [ ]:
# online preprocess == offline preprocess인지 확인
from utils.Mediapipe import MediaPipe
from preprocess.crop_data import crop_cheeks_from_detection_result

face_landmarker = MediaPipe()
img_path = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/_origin_data/028_data/data/1. 디지털카메라/0001/0001_01_F.jpg'
with open(img_path, 'rb') as f:
    image_content = f.read()
mp_image = face_landmarker.create_mp_image(image_content)

detection_result = face_landmarker.landmarker.detect(mp_image)
landmarks = detection_result.face_landmarks[0]

left_cheek, right_cheek = crop_cheeks_from_detection_result(landmarks, mp_image)

with open('/home/work/hocheol_dir/workspace/datasets/v0.2.2_data/pigment/028_data/data/left_0001_01_F.png', 'rb') as f:
    left_cheek2 = face_landmarker.create_mp_image(f.read()).numpy_view()
np.array_equal(left_cheek, left_cheek2)

In [ ]:
import os

source_dirs = ['/home/work/hocheol_dir/workspace/datasets/_origin_data/028_data/data/1. 디지털카메라', '/home/work/hocheol_dir/workspace/datasets/_origin_data/028_data/data/2. 스마트패드']
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/028_data/data'

path_groups = []

for source_dir in source_dirs:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(('_L.jpg', '_L15.jpg', '_L30.jpg')):
                source_path = os.path.join(root, file)
                left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
                path_groups.append((source_path, left_path))

path_groups.sort()
print(len(path_groups))

In [ ]:
path_groups[0]

In [ ]:
# 왼쪽
import sys
sys.path.append('/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_left
from tqdm import tqdm
import multiprocessing
from multiprocessing import Pool, cpu_count

multiprocessing.set_start_method('spawn', force=True)
num_workers = 32

with Pool(num_workers) as pool:
    fail_list = list(tqdm(pool.imap(process_left, path_groups), total=len(path_groups)))

fail_paths = set(filter(None, fail_list))
print(f"Failed to process {len(fail_paths)} images.")

In [5]:
import os

source_dirs = ['/home/work/hocheol_dir/workspace/datasets/_origin_data/028_data/data/1. 디지털카메라', '/home/work/hocheol_dir/workspace/datasets/_origin_data/028_data/data/2. 스마트패드']
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/028_data/data'

path_groups = []

for source_dir in source_dirs:
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(('_R.jpg', '_R15.jpg', '_R30.jpg')):
                source_path = os.path.join(root, file)
                right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
                path_groups.append((source_path, right_path))

path_groups.sort()
print(len(path_groups))

2895


In [6]:
path_groups[0]

('/home/work/hocheol_dir/workspace/datasets/_origin_data/028_data/data/1. 디지털카메라/0001/0001_01_R15.jpg',
 '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/028_data/data/right_0001_01_R15.png')

In [ ]:
import sys
sys.path.append('/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_right, init_worker
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

num_workers = 32

def parallel_process_R(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers, initializer=init_worker) as executor:
        fail_list = list(tqdm(executor.map(process_right, group), total=len(group)))
    return set(filter(None, fail_list))
fail_paths = parallel_process_R(path_groups, num_workers)
print(f"Failed to process {len(fail_paths)} R images")

### 030

In [4]:
import os
from preprocess.crop_data import crop_cheeks_from_image

source_dir = '/home/work/hocheol_dir/workspace/datasets/_origin_data/030_data/data'
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/030_data/data'

exts = set()

for root, dirs, files in os.walk(source_dir):
    for file in files:
        exts.add(file.split('.')[-1])

print(exts)

{'JPG', 'jpg'}


In [5]:
path_groups = []

for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.lower().endswith('.jpg'):
            source_path = os.path.join(root, file)
            left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
            right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
            path_groups.append((source_path, left_path, right_path))

path_groups.sort()
print(len(path_groups))

1624


In [ ]:
import sys
sys.path.insert(0, '/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_one
from tqdm import tqdm
import multiprocessing
from multiprocessing import Pool

multiprocessing.set_start_method('spawn', force=True)
num_workers = 64
os.makedirs(os.path.dirname(path_groups[0][-1]), exist_ok=True)

with Pool(num_workers) as pool:
    fail_list = list(tqdm(pool.imap(process_one, path_groups), total=len(path_groups)))

fail_paths = set(filter(None, fail_list))
print(f"Failed to process {len(fail_paths)} images.")

In [ ]:
# online preprocess == offline preprocess인지 확인
from utils.Mediapipe import MediaPipe
from preprocess.crop_data import crop_cheeks_from_detection_result
import numpy as np

face_landmarker = MediaPipe()
with open(path_groups[0][0], 'rb') as f:
    image_content = f.read()
mp_image = face_landmarker.create_mp_image(image_content)

detection_result = face_landmarker.landmarker.detect(mp_image)
landmarks = detection_result.face_landmarks[0]

left_cheek, right_cheek = crop_cheeks_from_detection_result(landmarks, mp_image)

with open(path_groups[0][1], 'rb') as f:
    left_cheek2 = face_landmarker.create_mp_image(f.read()).numpy_view()
np.array_equal(left_cheek, left_cheek2)

### 034

In [17]:
import os
from preprocess.crop_data import crop_cheeks_from_image

source_dir = '/home/work/hocheol_dir/workspace/datasets/_origin_data/034_data/data'
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/034_data/data'

exts = set()

for root, dirs, files in os.walk(source_dir):
    for file in files:
        exts.add(file.split('.')[-1])

print(exts)

{'JPG', 'jpg'}


In [18]:
path_groups = []

for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.lower().endswith('.jpg'):
            source_path = os.path.join(root, file)
            left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
            right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
            path_groups.append((source_path, left_path, right_path))

path_groups.sort()
print(len(path_groups))

5400


In [ ]:
import sys
sys.path.insert(0, '/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_one
from tqdm import tqdm
import multiprocessing
from multiprocessing import Pool

multiprocessing.set_start_method('spawn', force=True)
num_workers = 64
os.makedirs(os.path.dirname(path_groups[0][-1]), exist_ok=True)

with Pool(num_workers) as pool:
    fail_list = list(tqdm(pool.imap(process_one, path_groups), total=len(path_groups)))

fail_paths = set(filter(None, fail_list))
print(f"Failed to process {len(fail_paths)} images.")

In [23]:
# online preprocess == offline preprocess인지 확인
from utils.Mediapipe import MediaPipe
from preprocess.crop_data import crop_cheeks_from_detection_result
import numpy as np

face_landmarker = MediaPipe()
with open(path_groups[0][0], 'rb') as f:
    image_content = f.read()
mp_image = face_landmarker.create_mp_image(image_content)

detection_result = face_landmarker.landmarker.detect(mp_image)
landmarks = detection_result.face_landmarks[0]

left_cheek, right_cheek = crop_cheeks_from_detection_result(landmarks, mp_image)

with open(path_groups[0][1], 'rb') as f:
    left_cheek2 = face_landmarker.create_mp_image(f.read()).numpy_view()
if not np.array_equal(left_cheek, left_cheek2):
    raise Exception("Cropped left cheek does not match the saved image.")
np.array_equal(left_cheek, left_cheek2)

I0000 00:00:1751946036.998196 2185582 task_runner.cc:85] GPU suport is not available: INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:84) egl_initializedUnable to initialize EGL
W0000 00:00:1751946037.002169 2185582 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1751946037.007862 2343124 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1751946037.023070 2343127 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/work/hocheol_dir/.train/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is d

True

### 045

In [1]:
import os

source_dir = '/home/work/hocheol_dir/workspace/datasets/_origin_data/045_data/data'
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/045_data/data'

exts = set()

for root, dirs, files in os.walk(source_dir):
    for file in files:
        exts.add(file.split('.')[-1])

print(exts)

{'jpg', 'JPG'}


In [2]:
F_groups = []
L_groups = []
R_groups = []

for file in os.listdir(source_dir):
    if file.lower().endswith('.jpg'):
        source_path = os.path.join(source_dir, file)
        if '_0_' in file:
            left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
            right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
            F_groups.append((source_path, left_path, right_path))
        elif '_-45_' in file:
            left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
            L_groups.append((source_path, left_path, ))
        elif '_45_' in file:
            right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
            R_groups.append((source_path, right_path))
        
F_groups.sort()
L_groups.sort()
R_groups.sort()
len(F_groups), len(L_groups), len(R_groups), sum([len(F_groups), len(L_groups), len(R_groups)])

(1819, 1407, 1396, 4622)

In [ ]:
import sys
sys.path.insert(0, '/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_one, process_left, process_right
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

num_workers = os.cpu_count() - 1
os.makedirs(os.path.dirname(F_groups[0][-1]), exist_ok=True)
def parallel_process_F(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        fail_list = list(tqdm(executor.map(process_one, group), total=len(group)))
    return set(filter(None, fail_list))

def parallel_process_L(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        fail_list = list(tqdm(executor.map(process_left, group), total=len(group)))
    return set(filter(None, fail_list))

def parallel_process_R(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        fail_list = list(tqdm(executor.map(process_right, group), total=len(group)))
    return set(filter(None, fail_list))

# fail_F = parallel_process_F(F_groups, num_workers)
fail_L = parallel_process_L(L_groups, num_workers)
fail_R = parallel_process_R(R_groups, num_workers)

fail_paths = fail_F.union(fail_L).union(fail_R)

print(f"Failed to process {len(fail_paths)} images.")

In [ ]:
failed_paths= fail_L.union(fail_R)
failed_paths

### 118_data (고화질 only)

In [1]:
import os

source_dir = '/home/work/hocheol_dir/workspace/datasets/_origin_data/118_data/data'
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/118_data/data'

exts = set()

for root, dirs, files in os.walk(source_dir):
    for file in files:
        exts.add(file.split('.')[-1])

print(exts)

{'png'}


In [2]:
F_groups = []

for file in os.listdir(source_dir):
    if file.lower().endswith('.png'):
        source_path = os.path.join(source_dir, file)
        
        left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
        right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
        F_groups.append((source_path, left_path, right_path))
        
F_groups.sort()
len(F_groups)

2484

In [3]:
len(F_groups)

2484

In [ ]:
import sys
sys.path.insert(0, '/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_one
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

num_workers = 64
os.makedirs(os.path.dirname(F_groups[0][-1]), exist_ok=True)

def parallel_process(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        fail_list = list(tqdm(executor.map(process_one, group), total=len(group)))
    
    return set(filter(None, fail_list))

fail_F = parallel_process(F_groups, num_workers)

print(f"Failed to process {len(fail_F)} images.")
print(fail_F)

In [5]:
len(fail_F)

3

In [7]:
fail_F

{'/home/work/hocheol_dir/workspace/datasets/_origin_data/118_data/data/0003_1988_20_00000047_D.png',
 '/home/work/hocheol_dir/workspace/datasets/_origin_data/118_data/data/0010_1997_11_00000037_D.png',
 '/home/work/hocheol_dir/workspace/datasets/_origin_data/118_data/data/0906_1985_33_00000085_D.png'}

### 999

In [1]:
import os

source_dir = '/home/work/hocheol_dir/workspace/datasets/_origin_data/999_data/data'
dest_dir = '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/999_data/data'

exts = set()

for root, dirs, files in os.walk(source_dir):
    for file in files:
        exts.add(file.split('.')[-1])

print(exts)

{'csv', 'jpeg', 'jpg', 'JPG', 'png'}


In [2]:
F_groups = []
L_groups = []
R_groups = []

for root, dirs, files in os.walk(source_dir):
    for file in files:
        if not file.lower().endswith('.csv'):
            source_path = os.path.join(root, file)
            if 'F' in file:
                left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
                right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
                F_groups.append((source_path, left_path, right_path))
            elif 'L' in file:
                left_path = f"{dest_dir}/left_{os.path.splitext(file)[0]}.png"
                L_groups.append((source_path, left_path, ))
            elif 'R' in file:
                right_path = f"{dest_dir}/right_{os.path.splitext(file)[0]}.png"
                R_groups.append((source_path, right_path))
        
F_groups.sort()
L_groups.sort()
R_groups.sort()
len(F_groups), len(L_groups), len(R_groups), sum([len(F_groups), len(L_groups), len(R_groups)])

(1996, 1896, 1934, 5826)

In [3]:
F_groups[0]

('/home/work/hocheol_dir/workspace/datasets/_origin_data/999_data/data/SNUH_final/31001_01_F.JPG',
 '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/999_data/data/left_31001_01_F.png',
 '/home/work/hocheol_dir/datasets/v0.2.2_data/pigment/999_data/data/right_31001_01_F.png')

In [ ]:
import sys
sys.path.insert(0, '/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from tqdm import tqdm
from utils.Mediapipe import MediaPipe
from crop_module import process_one, process_left, process_right, init_worker
from concurrent.futures import ProcessPoolExecutor

fail_paths = set()

face_landmarker = MediaPipe()
os.makedirs(os.path.dirname(F_groups[0][-1]), exist_ok=True)

num_workers = os.cpu_count() - 1

def parallel_process_F(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers, initializer=init_worker) as executor:
        fail_list = list(tqdm(executor.map(process_one, group), total=len(group)))
    return set(filter(None, fail_list))

fail_F = parallel_process_F(F_groups, num_workers)


print(f"Failed to process {len(fail_F)} F images")

In [ ]:
import sys
sys.path.append('/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_left, init_worker
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

num_workers = 8

def parallel_process_L(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers, initializer=init_worker) as executor:
        fail_list = list(tqdm(executor.map(process_left, group), total=len(group)))
    return set(filter(None, fail_list))

fail_L = parallel_process_L(L_groups, num_workers)
print(f"Failed to process {len(fail_L)} L images")

In [ ]:
import sys
sys.path.append('/home/work/hocheol_dir/workspace/preprocess/v0.2.2/pigment')
from crop_module import process_right, init_worker
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

num_workers = 32

def parallel_process_R(group, num_workers=64):
    with ProcessPoolExecutor(max_workers=num_workers, initializer=init_worker) as executor:
        fail_list = list(tqdm(executor.map(process_right, group), total=len(group)))
    return set(filter(None, fail_list))
fail_R = parallel_process_R(R_groups, num_workers)
print(f"Failed to process {len(fail_R)} R images")